# Predicting next-session cognitive fatigue
A reproducible replacement for `program.ipynb`.

**Question:** do current behavioral and physiological measurements improve prediction of next-session self-reported fatigue beyond current fatigue, session and experimental condition?

This notebook preserves the original M0–M3 definitions. It adds validated joins, documented feature selection, subject-level exclusions, an extreme-transition audit, multiple estimators, subject-separated predictions and a sensitivity case study. Run cells in order with **Restart Kernel → Run All**.

**Data status:** the source notebook was available, but its raw CSV files were absent when this replacement was prepared. No empirical scores, exclusions or predictions are claimed here. The notebook stops clearly if inputs are absent; it never substitutes synthetic observations. Saved source-notebook outputs are not treated as raw data.

## Analysis map
1. Configuration and input contracts
2. Session features and strictly adjacent temporal targets
3. Descriptive feature analysis
4. Auditable quality rules and feature selection
5. M0–M3 grouped out-of-fold predictions with four estimators
6. Sensitivity to exclusions and a held-out feature-perturbation case study
7. Exports and interpretation

**Prediction timing:** features must be available by the end of session t; the target is fatigue at t+1. This is next-session prediction, not evidence that fatigue is predicted before any conscious report. No future measurements or subject identifiers enter predictors.

## 1. Environment and configuration
Install dependencies in the selected kernel environment, for example:
```bash
python -m pip install numpy pandas scipy matplotlib seaborn scikit-learn jupyterlab
```
The default input location is `/data`. Set `DATA_DIR` to the folder containing the three named CSVs. Outputs go into a separate results directory; input files are never overwritten. Rules below are explicit analyst choices, not established clinical cutoffs. Change them before inspecting performance and report sensitivity results.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("/data")  # Change this if the CSV folder is mounted elsewhere.
RESULTS_DIR = Path("fatigue_results")
SEED = 42
N_SPLITS = 5
# Conservative robust-z rules; sensitivity cases below vary the burden cutoff.
OUTLIER_Z = 3.5
SUBJECT_BURDEN = 0.20
MIN_OBSERVED_CELLS = 8
EXTREME_TRANSITION_Z = 4.5
MAX_MISSING = 0.40
MAX_CORRELATION = 0.90
sns.set_theme(style="whitegrid", context="notebook")
FILES = {
    "ratings": "MRSFATIGUE_ratings_session_zenodo.csv",
    "trials": "MRSFATIGUE_ntask_session_zenodo.csv",
    "mrs": "MRSFATIGUE_MRS_semiLaser_session_ratio_Glu_tCr_zenodo.csv",
}
VERSIONS = {p: metadata.version(p) for p in
            ["numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn"]}
print("Python", platform.python_version(), VERSIONS)

### Input contracts and provenance
The required tables come from the original notebook. We deliberately load only the modalities used in M0–M3: ratings, n-task trials and semiLASER glutamate. DWS, pupil and choice parameters are outside this comparison; adding them requires a separate timing and missingness analysis.

Duplicate fatigue ratings are **not averaged** because within-session timing is unknown. All ambiguous subject × condition × session fatigue keys are excluded and logged. Duplicate MRS ROI keys and duplicate trial identifiers raise errors rather than silently changing the unit of analysis. Numeric conversions must succeed; infinity becomes missing and is audited.

In [ ]:
missing_files = [str(DATA_DIR / f) for f in FILES.values() if not (DATA_DIR / f).is_file()]
if missing_files:
    raise FileNotFoundError("Raw CSV files are required. Set DATA_DIR. Missing: " + ", ".join(missing_files))
raw = {name: pd.read_csv(DATA_DIR / file) for name, file in FILES.items()}
provenance = {name: {"file": str(DATA_DIR / file),
                    "sha256": hashlib.sha256((DATA_DIR / file).read_bytes()).hexdigest(),
                    "rows": len(raw[name])} for name, file in FILES.items()}
KEY = ["subject_id", "condition", "session"]
REQUIRED = {
    "ratings": KEY + ["question", "rating"],
    "trials": KEY + ["block", "trial", "performance", "response_time", "rule_switch"],
    "mrs": KEY + ["ROI", "Glu_tCr"],
}
for name, columns in REQUIRED.items():
    absent = set(columns) - set(raw[name].columns)
    if absent:
        raise ValueError(f"{name}: missing columns {sorted(absent)}")
    if raw[name][KEY].isna().any().any():
        raise ValueError(f"{name}: missing join keys")
    raw[name]["subject_id"] = raw[name]["subject_id"].astype(str)
    raw[name]["condition"] = raw[name]["condition"].astype(str).str.strip()
    raw[name]["session"] = pd.to_numeric(raw[name]["session"], errors="raise")
    if not (raw[name]["session"] % 1 == 0).all():
        raise ValueError("Session must be an integer ordinal, not a label with unknown order.")
    raw[name]["session"] = raw[name]["session"].astype(int)
for name, columns in {"ratings": ["rating"], "trials": ["performance", "response_time", "rule_switch"],
                      "mrs": ["Glu_tCr"]}.items():
    for column in columns:
        raw[name][column] = pd.to_numeric(raw[name][column], errors="raise").replace([np.inf, -np.inf], np.nan)
display(pd.DataFrame(provenance).T)

## 2. Build one row per subject, condition and session
Performance and rule switching must be binary (0/1). Nonpositive reaction times become missing; we do not invent a physiologically justified upper cutoff. Reaction time units remain those in the source file. Mean performance uses all nonmissing trials, including errors; reaction times use all valid positive times. Confirm these definitions against the study protocol before publication.

Within-session changes compare the first and second halves after sorting by block and trial. Grouping always includes condition, preventing cross-condition mixing. These end-of-session features cannot be used for a prediction made before that session finishes.

In [ ]:
trials = raw["trials"].copy()
for column in ["performance", "rule_switch"]:
    if not trials[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must be binary 0/1; verify source coding.")
if trials.duplicated(KEY + ["block", "trial"]).any():
    raise ValueError("Repeated trial keys: resolve task/trial identity before aggregation.")
invalid_rt = trials["response_time"].le(0)
print("Nonpositive reaction times set to missing:", int(invalid_rt.sum()))
trials.loc[invalid_rt, "response_time"] = np.nan
trials = trials.sort_values(KEY + ["block", "trial"])
group = trials.groupby(KEY, sort=False)
position = group.cumcount()
trials["half"] = np.where(position < group["trial"].transform("size") / 2, "early", "late")
behavior = group.agg(accuracy=("performance", "mean"), median_rt=("response_time", "median"),
                     mean_rt=("response_time", "mean"), rt_std=("response_time", "std"),
                     switch_fraction=("rule_switch", "mean"), n_trials=("trial", "size")).reset_index()
for half in ["early", "late"]:
    part = trials.loc[trials["half"].eq(half)].groupby(KEY).agg(
        **{f"{half}_accuracy": ("performance", "mean"), f"{half}_rt": ("response_time", "mean")}
    ).reset_index()
    behavior = behavior.merge(part, on=KEY, validate="one_to_one")
behavior["within_session_rt_change"] = behavior["late_rt"] - behavior["early_rt"]
behavior["within_session_accuracy_change"] = behavior["late_accuracy"] - behavior["early_accuracy"]

fatigue = raw["ratings"].loc[raw["ratings"]["question"].eq("Fatigue"), KEY + ["rating"]].copy()
ambiguous = fatigue.loc[fatigue.duplicated(KEY, keep=False)].copy()
fatigue = fatigue.loc[~fatigue.duplicated(KEY, keep=False)].rename(columns={"rating": "fatigue"})
# Both endpoints must have unambiguous ratings; a missing intermediate session cannot create a jump.
fatigue = fatigue.sort_values(KEY)
for source, dest in [("fatigue", "fatigue_future"), ("session", "session_future")]:
    fatigue[dest] = fatigue.groupby(["subject_id", "condition"])[source].shift(-1)
transitions = fatigue.loc[(fatigue["session_future"] - fatigue["session"]).eq(1)].dropna(
    subset=["fatigue", "fatigue_future"]).copy()
transitions["delta_fatigue"] = transitions["fatigue_future"] - transitions["fatigue"]

mrs = raw["mrs"].copy()
if mrs.duplicated(KEY + ["ROI"]).any():
    raise ValueError("Repeated MRS ROI keys: do not average unknown repeats.")
mrs_wide = mrs.pivot(index=KEY, columns="ROI", values="Glu_tCr")
mrs_wide.columns = [f"MRS_Glu_tCr_{roi}" for roi in mrs_wide.columns]
mrs_wide = mrs_wide.reset_index()
if "MRS_Glu_tCr_PFC" not in mrs_wide:
    raise ValueError("PFC glutamate is required for M2 and M3.")
sessions = fatigue[KEY + ["fatigue"]].merge(behavior, on=KEY, how="left", validate="one_to_one")
sessions = sessions.merge(mrs_wide, on=KEY, how="left", validate="one_to_one")
model_data = transitions.merge(behavior, on=KEY, how="left", validate="one_to_one").merge(
    mrs_wide, on=KEY, how="left", validate="one_to_one").reset_index(drop=True)
model_data["row_id"] = np.arange(len(model_data))
assert not model_data.duplicated(KEY).any()
assert (model_data["session_future"] == model_data["session"] + 1).all()
print("Ambiguous rating rows excluded:", len(ambiguous))
print("Transitions:", len(model_data), "; subjects:", model_data["subject_id"].nunique())
display(model_data.groupby(["session", "session_future"]).size().rename("n").reset_index())

## 3. Feature definitions and descriptive analysis
M0 uses current fatigue, session (linear ordinal trend) and condition. M1 adds behavior; M2 adds PFC Glu/tCr; M3 combines both. This preserves the original comparisons. Glu/tCr is a metabolite ratio, not an absolute glutamate concentration.

The broader behavioral candidate pool allows redundant measures to be assessed. A deterministic, training-only selector below removes sparse, constant and highly correlated optional features. M0 covariates are always retained. A feature group can therefore reduce to its baseline if no optional feature is usable; the selection audit makes that visible.

Plots in this section are **descriptive full-data summaries**. They never determine thresholds or selected features. Repeated observations are not independent; correlations are not causal effects or inferential tests.

In [ ]:
BASE = ["fatigue", "session", "condition"]
BEHAVIOR = ["accuracy", "median_rt", "within_session_rt_change", "within_session_accuracy_change"]
CANDIDATES = BEHAVIOR + ["mean_rt", "rt_std", "switch_fraction", "early_accuracy", "late_accuracy", "early_rt", "late_rt"]
GLU = "MRS_Glu_tCr_PFC"
FEATURES = {"M0": BASE, "M1": BASE + BEHAVIOR, "M2": BASE + [GLU], "M3": BASE + BEHAVIOR + [GLU]}
AUDIT_FEATURES = BEHAVIOR + [GLU]  # Fixed, nonredundant pool avoids counting the same signal repeatedly.
DESCRIPTIVE = ["fatigue"] + CANDIDATES + [GLU]
feature_summary = sessions[DESCRIPTIVE].describe().T
feature_summary["missing_fraction"] = sessions[DESCRIPTIVE].isna().mean()
display(feature_summary.round(3))
# matplotlib: individual trajectories expose abrupt changes hidden by a group mean.
fig, ax = plt.subplots(figsize=(10, 4))
for _, part in fatigue.groupby(["subject_id", "condition"]):
    ax.plot(part["session"], part["fatigue"], alpha=0.25, marker=".")
ax.set(xlabel="Session", ylabel="Fatigue (source scale)", title="Individual fatigue trajectories")
plt.show()
# seaborn: rank correlations tolerate non-Gaussian feature distributions.
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(sessions[DESCRIPTIVE].corr(method="spearman"), center=0, vmin=-1, vmax=1,
            cmap="vlag", ax=ax)
ax.set_title("Descriptive Spearman feature correlations")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=model_data, x="delta_fatigue", hue="condition", element="step", ax=axes[0])
sns.scatterplot(data=model_data, x=GLU, y="delta_fatigue", hue="condition", ax=axes[1])
plt.tight_layout(); plt.show()
# Third graphic: scipy's normal probability plot diagnoses heavy tails, not independence or causality.
fig, ax = plt.subplots(figsize=(6, 4))
if len(model_data) >= 3:
    stats.probplot(model_data["delta_fatigue"], dist="norm", plot=ax)
ax.set_title("SciPy Q–Q plot: fatigue transitions")
plt.show()

## 4. Outlier rules and the suspected extreme transition
**Subjects with many outliers:** fit median/MAD limits to training session features only. A cell is flagged when its absolute robust z-score exceeds 3.5; when MAD is zero, use IQR/1.349, and if both are zero use an infinite scale (no statistically supported spread estimate). Missing cells are not outliers. A subject is removed from training when more than 20% of observed cells are flagged, provided at least eight cells are observed. All conditions for that subject are removed. Scores below the minimum are labeled as insufficient evidence, not proof of quality.

**Extreme transition:** inspect the absolute robust z-score of `delta_fatigue`. Remove at most one transition from each training partition, and only if it exceeds 4.5. The largest change is not deleted automatically. Ties resolve deterministically by row_id. If several exceed the threshold, the audit reports that fact; the “one extreme” assumption needs review. No residual-based deletion is used.

The descriptive audit below uses all available data to show which records a full-data refit would exclude. Honest validation recomputes rules within each training fold and **retains every test observation**, including extreme transitions. This prevents selecting an easy test population using its outcomes. Exclusions change the training population, and unusual observations may still be valid biology.

In [ ]:
def robust_location_scale(frame):
    """Fit robust feature limits without using future fatigue or test participants."""
    center = frame.median()
    scale = 1.4826 * frame.sub(center).abs().median()
    fallback = (frame.quantile(.75) - frame.quantile(.25)) / 1.349
    scale = scale.where(scale.gt(0), fallback)
    return center, scale.where(scale.gt(0), np.inf)


def filter_training(train, session_pool, burden=SUBJECT_BURDEN, remove_extreme=True):
    """Return retained training transitions and explicit subject/transition audit tables."""
    # Use only sessions belonging to training subjects; no outer-test session affects limits.
    pool = session_pool.loc[session_pool.subject_id.isin(train.subject_id.unique())].copy()
    center, scale = robust_location_scale(pool[AUDIT_FEATURES])
    observed = pool[AUDIT_FEATURES].notna()
    flags = pool[AUDIT_FEATURES].sub(center).div(scale).abs().gt(OUTLIER_Z) & observed
    counts = pd.DataFrame({"observed": observed.sum(axis=1), "flagged": flags.sum(axis=1),
                           "subject_id": pool.subject_id}).groupby("subject_id")[["observed", "flagged"]].sum()
    counts["burden"] = counts.flagged / counts.observed.replace(0, np.nan)
    counts["enough_evidence"] = counts.observed.ge(MIN_OBSERVED_CELLS)
    counts["excluded"] = counts.enough_evidence & counts.burden.gt(burden)
    rejected = counts.index[counts.excluded] if burden < 1 else []
    kept = train.loc[~train.subject_id.isin(rejected)].copy()
    transition_audit = kept[["row_id"] + KEY + ["delta_fatigue"]].copy()
    c, s = robust_location_scale(kept[["delta_fatigue"]])
    transition_audit["robust_z"] = kept.delta_fatigue.sub(c.iloc[0]).div(s.iloc[0]).abs()
    transition_audit["candidate"] = transition_audit.robust_z.gt(EXTREME_TRANSITION_Z)
    transition_audit["excluded"] = False
    candidates = transition_audit.loc[transition_audit.candidate].sort_values(
        ["robust_z", "row_id"], ascending=[False, True])
    if remove_extreme and len(candidates):
        row_id = candidates.iloc[0].row_id
        transition_audit.loc[transition_audit.row_id.eq(row_id), "excluded"] = True
        kept = kept.loc[kept.row_id.ne(row_id)]
    return kept, counts.reset_index(), transition_audit

full_clean, subject_audit, transition_audit = filter_training(model_data, sessions)
display(subject_audit.sort_values("burden", ascending=False))
display(transition_audit.sort_values("robust_z", ascending=False).head(10))
print("Full-data refit subject exclusions:", int(subject_audit.excluded.sum()))
print("Extreme candidates:", int(transition_audit.candidate.sum()),
      "; removed:", int(transition_audit.excluded.sum()))

### Training-only feature selection
This selector retains optional features with at most 40% missingness and nonzero variation, then greedily removes an optional feature if its absolute Spearman correlation with an already retained numeric feature exceeds 0.90. Column order expresses the prespecified preference (accuracy and median RT before redundant alternatives). It does not search for the subset with the best test score.

Imputation uses training medians; an entirely missing protected numeric covariate falls back to zero and is logged by its availability summary. Scaling uses training mean/variance. Condition is one-hot encoded with unknown levels ignored. These steps are refitted for every fold. Hyperparameters are fixed in advance; tuning would require an inner subject-grouped validation loop.

In [ ]:
class FoldFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        self.selected_ = [c for c in BASE if c in self.features]
        self.audit_ = []
        for column in self.features:
            if column in self.selected_:
                continue
            reason = "retained"
            if X[column].isna().mean() > MAX_MISSING:
                reason = "too_many_missing"
            elif X[column].nunique(dropna=True) <= 1:
                reason = "constant_or_empty"
            else:
                prior = [c for c in self.selected_ if c != "condition"]
                corr = X[prior + [column]].corr(method="spearman")[column].drop(column).abs()
                if corr.gt(MAX_CORRELATION).any():
                    reason = "redundant_with_" + str(corr.idxmax())
            if reason == "retained":
                self.selected_.append(column)
            self.audit_.append({"feature": column, "decision": reason})
        self.numeric_ = [c for c in self.selected_ if c != "condition"]
        self.medians_ = X[self.numeric_].median().fillna(0)
        return self

    def transform(self, X):
        out = X[self.selected_].copy()
        out[self.numeric_] = out[self.numeric_].fillna(self.medians_)
        return out


def numeric_columns(frame):
    return [c for c in frame.columns if c != "condition"]


def make_model(features, estimator):
    algorithms = {
        "OLS": LinearRegression(),
        "Ridge": Ridge(alpha=10),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000, random_state=SEED),
        "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=3,
                                               max_features=1.0, random_state=SEED, n_jobs=-1),
    }
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_columns),
        ("condition", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), ["condition"]),
    ])
    return Pipeline([("selection", FoldFeatureSelector(features)),
                     ("preprocessing", preprocessing), ("regression", algorithms[estimator])])

# Broader candidate-pool selection is diagnostic; M0–M3 retain their prespecified pools.
candidate_selector = FoldFeatureSelector(BASE + CANDIDATES + [GLU]).fit(full_clean)
display(pd.DataFrame(candidate_selector.audit_))

## 5. M0–M3 predictions and model comparison
The primary comparison uses the **same complete-case cohort and identical outer folds** across M0–M3 and all four estimators. This isolates feature-group differences rather than sample composition. Complete-case analysis can still be biased if missingness is informative; the sensitivity section repeats evaluation with imputation on the larger cohort.

Every subject appears in one test fold only. The target is next-session fatigue; predicted change is predicted future fatigue minus observed current fatigue. The persistence comparator predicts no change. OLS is an unregularized reference; Ridge and Elastic Net stabilize small-sample fits; Random Forest allows nonlinearities. No model is selected by a separate held-out test set, so the best observed OOF score is exploratory.

Subject exclusion and the extreme-transition removal affect only training data. Prediction tables preserve all held-out rows. Unbounded regression predictions are not clipped to the rating range, so error metrics reflect the actual estimator.

In [ ]:
ALL_FEATURES = list(dict.fromkeys(sum(FEATURES.values(), [])))
common = model_data.dropna(subset=ALL_FEATURES + ["fatigue_future"]).copy().reset_index(drop=True)
if common.subject_id.nunique() < 3:
    raise ValueError("At least three subjects with complete M0–M3 data are needed.")
folds = list(GroupKFold(n_splits=min(N_SPLITS, common.subject_id.nunique())).split(
    common, groups=common.subject_id))
display(pd.DataFrame([
    {"cohort": "all adjacent transitions", "rows": len(model_data), "subjects": model_data.subject_id.nunique()},
    {"cohort": "common complete cases", "rows": len(common), "subjects": common.subject_id.nunique()},
]))


def metrics(y, prediction):
    return {"MAE": mean_absolute_error(y, prediction),
            "RMSE": np.sqrt(mean_squared_error(y, prediction)),
            "R2": r2_score(y, prediction) if len(y) > 1 and np.var(y) > 0 else np.nan}


def evaluate(cohort, split_list, label, burden=SUBJECT_BURDEN, remove_extreme=True,
             algorithms=("OLS", "Ridge", "ElasticNet", "RandomForest")):
    predictions, audits, fold_metrics = [], [], []
    for fold, (train_idx, test_idx) in enumerate(split_list, 1):
        train, test = cohort.iloc[train_idx], cohort.iloc[test_idx]
        assert set(train.subject_id).isdisjoint(test.subject_id)
        cleaned, subject_log, transition_log = filter_training(train, sessions, burden, remove_extreme)
        if len(cleaned) < 5 or cleaned.subject_id.nunique() < 2:
            raise ValueError(f"{label}, fold {fold}: too little training data after exclusions.")
        for name, features in FEATURES.items():
            for algorithm in algorithms:
                model = make_model(features, algorithm).fit(cleaned[features], cleaned.fatigue_future)
                pred = model.predict(test[features])
                part = test[["row_id"] + KEY + ["fatigue", "fatigue_future", "delta_fatigue"]].copy()
                part["scenario"], part["fold"], part["model"], part["algorithm"] = label, fold, name, algorithm
                part["prediction"] = pred
                part["predicted_delta"] = pred - part.fatigue
                part["residual"] = part.fatigue_future - pred
                predictions.append(part)
                selection = model.named_steps["selection"]
                audits.append({"scenario": label, "fold": fold, "model": name, "algorithm": algorithm,
                               "n_train_before": len(train), "n_train_after": len(cleaned),
                               "excluded_subjects": subject_log.loc[subject_log.excluded, "subject_id"].tolist(),
                               "excluded_rows": transition_log.loc[transition_log.excluded, "row_id"].tolist(),
                               "selected_features": selection.selected_, "selection_decisions": selection.audit_})
                fold_metrics.append({"scenario": label, "fold": fold, "model": name, "algorithm": algorithm,
                                     "n_test": len(test), **metrics(test.fatigue_future, pred)})
    predictions = pd.concat(predictions, ignore_index=True)
    assert not predictions.duplicated(["scenario", "model", "algorithm", "row_id"]).any()
    scores = []
    for (name, algorithm), part in predictions.groupby(["model", "algorithm"]):
        assert len(part) == len(cohort)
        # Subject-balanced MAE gives each participant equal weight despite repeated transitions.
        subject_mae = part.assign(ae=part.residual.abs()).groupby("subject_id").ae.mean().mean()
        scores.append({"scenario": label, "model": name, "algorithm": algorithm, "n": len(part),
                       "subjects": part.subject_id.nunique(), "subject_balanced_MAE": subject_mae,
                       **metrics(part.fatigue_future, part.prediction)})
    return predictions, pd.DataFrame(scores), pd.DataFrame(audits), pd.DataFrame(fold_metrics)

predictions, scores, fold_audit, fold_scores = evaluate(common, folds, "primary")
persistence_score = metrics(common.fatigue_future, common.fatigue)
display(scores.sort_values("RMSE").round(3))
print("Persistence (future fatigue = current fatigue):", persistence_score)
display(predictions.head(12))
display(fold_audit.head(8))

### Prediction diagnostics and uncertainty
Inspect scatter and residual plots rather than relying only on R². Negative OOF R² means the errors exceed the pooled observed-target variance benchmark; it is not a software error. Bootstrap participants, not individual rows, to preserve within-person dependence. These intervals describe resampling uncertainty in fixed OOF predictions; they do not include model-refitting or feature-selection uncertainty and are not individual prediction intervals.

In [ ]:
ridge_predictions = predictions.loc[predictions.algorithm.eq("Ridge")]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col, name in enumerate(FEATURES):
    part = ridge_predictions.loc[ridge_predictions.model.eq(name)]
    axes[0, col].scatter(part.fatigue_future, part.prediction, alpha=.7)
    low, high = part[["fatigue_future", "prediction"]].min().min(), part[["fatigue_future", "prediction"]].max().max()
    axes[0, col].plot([low, high], [low, high], "k--")
    axes[0, col].set(title=f"Ridge {name}", xlabel="Observed future fatigue", ylabel="OOF prediction")
    axes[1, col].scatter(part.prediction, part.residual, alpha=.7)
    axes[1, col].axhline(0, color="black", linestyle="--")
    axes[1, col].set(xlabel="OOF prediction", ylabel="Observed − predicted")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
bootstrap_rows = []
wide = ridge_predictions.pivot(index="row_id", columns="model", values="prediction")
reference = common.set_index("row_id").loc[wide.index]
subject_ids = reference.subject_id.unique()
for repeat in range(500):
    sampled = rng.choice(subject_ids, len(subject_ids), replace=True)
    # Concatenation preserves multiplicity when a participant is sampled repeatedly.
    idx = np.concatenate([np.flatnonzero(reference.subject_id.to_numpy() == sid) for sid in sampled])
    y = reference.fatigue_future.to_numpy()[idx]
    base_rmse = np.sqrt(mean_squared_error(y, wide.M0.to_numpy()[idx]))
    for name in ["M1", "M2", "M3"]:
        difference = np.sqrt(mean_squared_error(y, wide[name].to_numpy()[idx])) - base_rmse
        bootstrap_rows.append({"repeat": repeat, "model": name, "RMSE_minus_M0": difference})
bootstrap = pd.DataFrame(bootstrap_rows)
display(bootstrap.groupby("model").RMSE_minus_M0.quantile([.025, .5, .975]).unstack())
print("Negative RMSE_minus_M0 favors the augmented feature group; intervals are exploratory.")

## 6. Sensitivity analysis (the requested “sensibility” study)
First repeat all M0–M3 Ridge predictions using the **same held-out rows and folds** under: no exclusions, subject exclusions only, extreme-transition exclusion only, and stricter/looser subject burden thresholds. Keeping the evaluation cohort fixed makes differences attributable to the training policy. The primary results already include both exclusions. All scenarios export row-level predictions.

A separate available-case analysis includes participants with missing optional predictors and fits median imputation/feature selection inside each fold. Its scores describe a different cohort and must not be read as a direct improvement over complete cases. No outcome or required join key is imputed.

In [ ]:
sensitivity_specs = [
    ("no_exclusions", 1.0, False),
    ("subjects_only", SUBJECT_BURDEN, False),
    ("extreme_only", 1.0, True),
    ("stricter_subject_rule", 0.10, True),
    ("looser_subject_rule", 0.30, True),
]
sensitivity_predictions, sensitivity_scores, sensitivity_audits = [], [], []
for label, burden, extreme in sensitivity_specs:
    p, s, a, _ = evaluate(common, folds, label, burden, extreme, algorithms=("Ridge",))
    sensitivity_predictions.append(p); sensitivity_scores.append(s); sensitivity_audits.append(a)
sensitivity_predictions = pd.concat(sensitivity_predictions, ignore_index=True)
sensitivity_scores = pd.concat(sensitivity_scores, ignore_index=True)
sensitivity_audits = pd.concat(sensitivity_audits, ignore_index=True)
display(sensitivity_scores.pivot(index="scenario", columns="model", values="RMSE").round(3))

available = model_data.dropna(subset=BASE + ["fatigue_future"]).copy().reset_index(drop=True)
available_folds = list(GroupKFold(n_splits=min(N_SPLITS, available.subject_id.nunique())).split(
    available, groups=available.subject_id))
available_predictions, available_scores, available_audit, _ = evaluate(
    available, available_folds, "available_case_imputation", algorithms=("Ridge",))
display(available_scores.round(3))

### Held-out case study: response to plausible feature changes
Select the lowest-row-id participant observation in the first test fold **without looking at its outcome or error**. Fit each M0–M3 Ridge model on the cleaned training fold only. Predict the actual case, then perturb accuracy, median RT or PFC Glu/tCr one at a time by ±0.5 training standard deviations. Clip scenarios to training observed bounds and accuracy to [0,1], recording actual values. The future-fatigue outcome is not changed or used in construction.

These are model response scenarios, not causal interventions or new measured subjects. Some combinations may be physiologically implausible, and a selected-out feature correctly has no effect. The baseline prediction is out of subject; scenario predictions have no observed ground truth. Report prediction changes rather than claiming scenario accuracy.

In [ ]:
train_idx, test_idx = folds[0]
case_train, _, _ = filter_training(common.iloc[train_idx], sessions)
case = common.iloc[test_idx].sort_values("row_id").iloc[[0]].copy()
assert case.subject_id.iloc[0] not in set(case_train.subject_id)
scenarios = [("observed_case", None, 0.0, case.copy())]
for feature in ["accuracy", "median_rt", GLU]:
    sd = case_train[feature].std(ddof=0)
    for sign in [-1, 1]:
        changed = case.copy()
        low, high = case_train[feature].min(), case_train[feature].max()
        if feature == "accuracy":
            low, high = max(0, low), min(1, high)
        changed[feature] = np.clip(case[feature].iloc[0] + sign * .5 * sd, low, high)
        scenarios.append((f"{feature}_{sign:+d}_half_sd", feature, sign * .5, changed))
case_rows = []
for name, features in FEATURES.items():
    fitted = make_model(features, "Ridge").fit(case_train[features], case_train.fatigue_future)
    original = fitted.predict(case[features])[0]
    for label, feature, sd_shift, frame in scenarios:
        predicted = fitted.predict(frame[features])[0]
        case_rows.append({"model": name, "scenario": label, "subject_id": case.subject_id.iloc[0],
                          "row_id": int(case.row_id.iloc[0]), "changed_feature": feature,
                          "requested_sd_shift": sd_shift,
                          "original_value": case[feature].iloc[0] if feature else np.nan,
                          "scenario_value": frame[feature].iloc[0] if feature else np.nan,
                          "prediction": predicted, "predicted_delta": predicted - case.fatigue.iloc[0],
                          "change_from_observed_prediction": predicted - original,
                          "feature_retained": feature in fitted.named_steps["selection"].selected_ if feature else True})
case_predictions = pd.DataFrame(case_rows)
display(case[KEY + ["fatigue", "fatigue_future"]])
display(case_predictions.round(4))
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=case_predictions.loc[case_predictions.scenario.ne("observed_case")],
            x="scenario", y="change_from_observed_prediction", hue="model", ax=ax)
ax.tick_params(axis="x", rotation=35)
ax.set(ylabel="Change in predicted future fatigue", title="Held-out sensitivity case study")
plt.tight_layout(); plt.show()

## 7. Final refits, exports and interpretation
Full-data refits are useful for future prediction; their fitted values are **not** generalization scores. The function below checks a new table and returns future-fatigue and fatigue-change predictions for every M0–M3 group. Use the same feature engineering and measurement timing for future subjects.

Exports include every OOF prediction, fold and exclusion audits, selection decisions, sensitivity cases, provenance and package versions. Do not distribute subject-level files without the study's data-sharing permissions. The source files remain unchanged.

In [ ]:
final_models = {}
refit_training, final_subject_log, final_transition_log = filter_training(common, sessions)
for name, features in FEATURES.items():
    final_models[name] = make_model(features, "Ridge").fit(refit_training[features], refit_training.fatigue_future)


def predict_new(new_features):
    """Predict engineered rows with all M0–M3 models; no future label is required."""
    absent = set(ALL_FEATURES) - set(new_features.columns)
    if absent:
        raise ValueError(f"Missing feature columns: {sorted(absent)}")
    if new_features[BASE].isna().any().any():
        raise ValueError("Current fatigue, session and condition must be observed.")
    result = new_features[BASE].copy()
    for name, fitted in final_models.items():
        result[f"{name}_future_fatigue"] = fitted.predict(new_features[FEATURES[name]])
        result[f"{name}_delta_fatigue"] = result[f"{name}_future_fatigue"] - new_features.fatigue
    return result


RESULTS_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    "feature_summary": feature_summary.reset_index(names="feature"),
    "ambiguous_ratings": ambiguous,
    "subject_audit_full_data": subject_audit,
    "final_refit_subject_audit": final_subject_log,
    "final_refit_transition_audit": final_transition_log,
    "transition_audit_full_data": transition_audit,
    "primary_oof_predictions": predictions,
    "primary_scores": scores,
    "fold_scores": fold_scores,
    "sensitivity_oof_predictions": sensitivity_predictions,
    "sensitivity_scores": sensitivity_scores,
    "available_case_oof_predictions": available_predictions,
    "available_case_scores": available_scores,
    "case_study_predictions": case_predictions,
    "bootstrap_ridge_rmse_differences": bootstrap,
}
for name, table in exports.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
# JSON preserves nested selected-feature lists and removal identities without string parsing.
for name, audit in {"primary_fold_audit": fold_audit, "sensitivity_fold_audit": sensitivity_audits,
                    "available_case_fold_audit": available_audit}.items():
    (RESULTS_DIR / f"{name}.json").write_text(audit.to_json(orient="records", indent=2))
manifest = {"versions": VERSIONS, "inputs": provenance, "seed": SEED,
            "rules": {"robust_z": OUTLIER_Z, "subject_burden": SUBJECT_BURDEN,
                      "min_observed_cells": MIN_OBSERVED_CELLS, "extreme_z": EXTREME_TRANSITION_Z,
                      "max_missing": MAX_MISSING, "max_correlation": MAX_CORRELATION},
            "feature_groups": FEATURES, "prediction_target": "fatigue at exactly session t+1",
            "evaluation": "subject-grouped OOF, all test observations retained",
            "persistence_metrics": persistence_score}
(RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("Results written to", RESULTS_DIR.resolve())
# Generated statements report results only after actual data execution.
ridge_scores = scores.loc[scores.algorithm.eq("Ridge")].set_index("model")
for name in FEATURES:
    print(f"Ridge {name}: OOF RMSE={ridge_scores.loc[name, 'RMSE']:.3f}; "
          f"MAE={ridge_scores.loc[name, 'MAE']:.3f}; R²={ridge_scores.loc[name, 'R2']:.3f}")

### How to interpret the completed run
- Compare M1, M2 and M3 against M0 using the same estimator, cohort and outer folds. Compare against persistence as well. Lower MAE/RMSE is better; higher R² is better. Inspect participant-balanced error and fold variation.
- A lower error after training exclusions supports a training policy under this sample and split, not a claim that excluded people or their fatigue changes were erroneous. Review the identities, feature burden and extreme-transition candidate count. If no candidate passes the prespecified threshold, this notebook deletes no transition.
- Examine whether conclusions change without exclusions, at alternate burden cutoffs, and when missing optional features are imputed. The available-case cohort answers a different population question.
- Selection frequency and an unchanged scenario response reveal whether an optional feature was used. A model's sensitivity to a feature does not prove that changing that feature changes fatigue.
- Small MRS cohorts, repeated measurements, possible temporal mismatch, missingness and threshold choices limit confidence. GroupKFold estimates performance for unseen subjects in the observed session schedule; it does not test deployment at later calendar times or a new study site. Whole-session features are available only at session end.
- The analysis has no external test set. Do not promote the best OOF model as independently validated. Confirm promising findings in a preregistered external cohort, with timing and measurement quality verified.

**Completion checklist:** inspect all audits; verify units and task timing against the protocol; document actual excluded IDs and transition keys; report complete-case attrition; present paired uncertainty and sensitivity results; retain raw data and the generated manifest.